# WEEK 3: HISTORICAL CUSTOMER LIFETIME VALUE & DOCUMENTATION
**Member:** Osonye Onyemazuwa (branch: hiren)  
**Project:** SaaS/E-Commerce Cohort Retention & CLTV Analysis  
**Organization:** Infotact Solutions  
**Date:** 24th – 27th June 2026

# Step 1: Import libraries and load dataset

In [1]:
import pandas as pd
import numpy as np
import os
print("Libraries imported succesfully")

df = pd.read_csv(r"C:\Users\onyer\Downloads\cleaned_retail_data_updated.csv")
print(f"\nDataset shape: {df.shape}")
print(f"\nColums: {df.columns.tolist()}")
df.head()

Libraries imported succesfully

Dataset shape: (392692, 11)

Colums: ['InvoiceNo', 'StockCode', 'Description', 'Quantity', 'InvoiceDate', 'UnitPrice', 'CustomerID', 'Country', 'TransactionMonth', 'CohortMonth', 'CohortIndex']


,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country,TransactionMonth,CohortMonth,CohortIndex
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,01-12-2010 08:26,2.55,17850,United Kingdom,01-12-2010,01-12-2010,0.0
1,536365,71053,WHITE METAL LANTERN,6,01-12-2010 08:26,3.39,17850,United Kingdom,01-12-2010,01-12-2010,0.0
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,01-12-2010 08:26,2.75,17850,United Kingdom,01-12-2010,01-12-2010,0.0
3,536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,01-12-2010 08:26,3.39,17850,United Kingdom,01-12-2010,01-12-2010,0.0
4,536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,01-12-2010 08:26,3.39,17850,United Kingdom,01-12-2010,01-12-2010,0.0


# Step 2: Calculate TotalRevenue per transaction

In [2]:
df["TotalRevenue"] = df["Quantity"] * df["UnitPrice"]

print("TotalRevenue Column added successfully")
print(df[["CustomerID", "InvoiceNo", "Quantity", "UnitPrice", "TotalRevenue"]])

TotalRevenue Column added successfully
        CustomerID  InvoiceNo  Quantity  UnitPrice  TotalRevenue
0            17850     536365         6       2.55         15.30
1            17850     536365         6       3.39         20.34
2            17850     536365         8       2.75         22.00
3            17850     536365         6       3.39         20.34
4            17850     536365         6       3.39         20.34
...            ...        ...       ...        ...           ...
392687       12680     581587        12       0.85         10.20
392688       12680     581587         6       2.10         12.60
392689       12680     581587         4       4.15         16.60
392690       12680     581587         4       4.15         16.60
392691       12680     581587         3       4.95         14.85

[392692 rows x 5 columns]


# Step 3: AOV

In [3]:
try:
    aov = pd.read_csv('outputs/siva_aov.csv')
    print("✅ siva AOV output loaded successfully")

except FileNotFoundError:
    print("⚠️ siva output not found - using fallback AOV calculation")
    
    customer_revenue = df.groupby('CustomerID')['TotalRevenue'].sum()
    customer_orders = df.groupby('CustomerID')['InvoiceNo'].nunique()
    aov = (customer_revenue / customer_orders).reset_index()
    aov.columns = ['CustomerID', 'AOV']

print(aov.head())

⚠️ siva output not found - using fallback AOV calculation
   CustomerID           AOV
0       12346  77183.600000
1       12347    615.714286
2       12348    449.310000
3       12349   1757.550000
4       12350    334.400000


# Step 4: Purchase frequency

In [4]:
try:
    purchase_freq = pd.read_csv('outputs/yash_frequency.csv')
    print("✅ yash output loaded successfully")

except FileNotFoundError:
    print("⚠️ yash output not found - using fallback")
    
    purchase_freq = df.groupby('CustomerID')['InvoiceNo'].nunique().reset_index()
    purchase_freq.columns = ['CustomerID', 'PurchaseFrequency']

print(purchase_freq.head())

⚠️ yash output not found - using fallback
   CustomerID  PurchaseFrequency
0       12346                  1
1       12347                  7
2       12348                  4
3       12349                  1
4       12350                  1


# Step 5: Historical CLTV

In [5]:
# Merge AOV and Purchase Frequency
cltv_df = aov.merge(purchase_freq, on='CustomerID')

# CLTV = AOV x Purchase Frequency
cltv_df['CLTV'] = cltv_df['AOV'] * cltv_df['PurchaseFrequency']

print(cltv_df.head(10))
print(f"\nCLTV Summary:\n{cltv_df['CLTV'].describe()}")

   CustomerID           AOV  PurchaseFrequency      CLTV
0       12346  77183.600000                  1  77183.60
1       12347    615.714286                  7   4310.00
2       12348    449.310000                  4   1797.24
3       12349   1757.550000                  1   1757.55
4       12350    334.400000                  1    334.40
5       12352    313.255000                  8   2506.04
6       12353     89.000000                  1     89.00
7       12354   1079.400000                  1   1079.40
8       12355    459.400000                  1    459.40
9       12356    937.143333                  3   2811.43

CLTV Summary:
count      4338.000000
mean       2048.688081
std        8985.230220
min           3.750000
25%         306.482500
50%         668.570000
75%        1660.597500
max      280206.020000
Name: CLTV, dtype: float64


In [6]:
os.makedirs('outputs', exist_ok=True)
cltv_df.to_csv('outputs/hiren_cltv.csv', index=False)
print("Saved successfully")

Saved successfully


# Step 6: Project 12-Month CLTV

In [7]:
# First merge CohortMonth into cltv_df
cltv_df = cltv_df.merge(df[['CustomerID', 'CohortMonth']].drop_duplicates(), 
                          on='CustomerID', how='left')

# Project 12-Month CLTV per customer
cltv_df['CLTV_12Month'] = cltv_df['CLTV'] * 12

# Group by CohortMonth segment
cohort_cltv = cltv_df.groupby('CohortMonth').agg(
    Total_Customers=('CustomerID', 'count'),
    Avg_AOV=('AOV', 'mean'),
    Avg_Purchase_Frequency=('PurchaseFrequency', 'mean'),
    Avg_CLTV=('CLTV', 'mean'),
    Avg_12Month_CLTV=('CLTV_12Month', 'mean'),
    Total_12Month_CLTV=('CLTV_12Month', 'sum')
).round(2).reset_index()

print("12-Month CLTV Per Cohort Segment:")
print(cohort_cltv)
print(f"\nBest Performing Cohort:")
print(cohort_cltv.loc[cohort_cltv['Avg_12Month_CLTV'].idxmax()])
print(f"\nWorst Performing Cohort:")
print(cohort_cltv.loc[cohort_cltv['Avg_12Month_CLTV'].idxmin()])

12-Month CLTV Per Cohort Segment:
   CohortMonth  Total_Customers  Avg_AOV  Avg_Purchase_Frequency  Avg_CLTV  \
0   01-01-2011              178   519.62                    6.83   3829.94   
1   01-02-2011              190   366.42                    5.78   3816.53   
2   01-03-2011              234   360.42                    4.67   1925.85   
3   01-04-2011              232   357.86                    5.09   2020.16   
4   01-05-2011              253   376.48                    4.74   1765.75   
5   01-06-2011              184   421.60                    3.86   1775.42   
6   01-07-2011              135   355.27                    3.53   1283.86   
7   01-08-2011              136   461.44                    3.75   1870.28   
8   01-09-2011              166   368.21                    3.57   1547.65   
9   01-10-2011              232   366.32                    2.66    962.44   
10  01-11-2011              263   371.15                    2.48    917.03   
11  01-12-2010              64

## Step 7: Customer Segmentation
Segmenting customers into three value tiers based on
their 12-Month CLTV using percentile thresholds.

| Segment | Threshold |
|---|---|
| High Value | Top 25% (above 75th percentile) |
| Mid Value | Middle 50% |
| Low Value | Bottom 25% (below 25th percentile) |

In [8]:
# Define thresholds
high_threshold = cltv_df['CLTV_12Month'].quantile(0.75)
low_threshold = cltv_df['CLTV_12Month'].quantile(0.25)

# Segmentation function
def segment_customer(cltv):
    if cltv >= high_threshold:
        return 'High Value'
    elif cltv <= low_threshold:
        return 'Low Value'
    else:
        return 'Mid Value'

# Apply segmentation
cltv_df['Segment'] = cltv_df['CLTV_12Month'].apply(segment_customer)

print("Customer Segment Distribution:")
print(cltv_df['Segment'].value_counts())
print(f"\nHigh Value Threshold: £{high_threshold:,.2f}")
print(f"Low Value Threshold:  £{low_threshold:,.2f}")

Customer Segment Distribution:
Segment
Mid Value     2168
High Value    1085
Low Value     1085
Name: count, dtype: int64

High Value Threshold: £19,927.17
Low Value Threshold:  £3,677.79


## Step 8: Calculate Maximum Acceptable CAC
Customer Acquisition Cost (CAC) should not exceed
30% of the projected 12-Month CLTV to ensure profitability.

> **Max CAC = CLTV_12Month x 0.30**

### Why 30%?
Industry standard suggests CAC should not exceed
30% of CLTV to maintain healthy profit margins.  
A business spending more than 30% of CLTV on
acquisition will struggle to remain profitable.

In [9]:
# Calculate Max CAC
cltv_df['Max_CAC'] = cltv_df['CLTV_12Month'] * 0.30

# Summary by segment
print("CAC Summary by Segment:")
print(cltv_df.groupby('Segment')['Max_CAC'].mean().round(2))
print(f"\nOverall Average Max CAC: £{cltv_df['Max_CAC'].mean():,.2f}")

CAC Summary by Segment:
Segment
High Value    23350.53
Low Value       640.92
Mid Value      2750.56
Name: Max_CAC, dtype: float64

Overall Average Max CAC: £7,375.28


## Step 9: CLTV Summary by Cohort Segment
Final summary table showing all key metrics
grouped by CohortMonth for business reporting.

In [10]:
# Final CLTV summary by cohort
cohort_summary = cltv_df.groupby('CohortMonth').agg(
    Total_Customers=('CustomerID', 'count'),
    Avg_AOV=('AOV', 'mean'),
    Avg_Purchase_Frequency=('PurchaseFrequency', 'mean'),
    Avg_CLTV=('CLTV', 'mean'),
    Avg_12Month_CLTV=('CLTV_12Month', 'mean'),
    Avg_Max_CAC=('Max_CAC', 'mean')
).round(2).reset_index()

print("CLTV Summary by Cohort Segment:")
print(cohort_summary)

CLTV Summary by Cohort Segment:
   CohortMonth  Total_Customers  Avg_AOV  Avg_Purchase_Frequency  Avg_CLTV  \
0   01-01-2011              178   519.62                    6.83   3829.94   
1   01-02-2011              190   366.42                    5.78   3816.53   
2   01-03-2011              234   360.42                    4.67   1925.85   
3   01-04-2011              232   357.86                    5.09   2020.16   
4   01-05-2011              253   376.48                    4.74   1765.75   
5   01-06-2011              184   421.60                    3.86   1775.42   
6   01-07-2011              135   355.27                    3.53   1283.86   
7   01-08-2011              136   461.44                    3.75   1870.28   
8   01-09-2011              166   368.21                    3.57   1547.65   
9   01-10-2011              232   366.32                    2.66    962.44   
10  01-11-2011              263   371.15                    2.48    917.03   
11  01-12-2010              644 

In [11]:
# Best and worst cohorts
best_cohort = cohort_summary.loc[cohort_summary['Avg_12Month_CLTV'].idxmax()]
worst_cohort = cohort_summary.loc[cohort_summary['Avg_12Month_CLTV'].idxmin()]

print("🏆 Best Performing Cohort:")
print(best_cohort)
print(f"\n⚠️ Worst Performing Cohort:")
print(worst_cohort)
print(f"\n📊 CLTV Gap between Best and Worst Cohort:")
print(f"£{best_cohort['Avg_12Month_CLTV'] - worst_cohort['Avg_12Month_CLTV']:,.2f}")

🏆 Best Performing Cohort:
CohortMonth               01-12-2010
Total_Customers                  644
Avg_AOV                       376.44
Avg_Purchase_Frequency         10.34
Avg_CLTV                     5346.26
Avg_12Month_CLTV            64155.13
Avg_Max_CAC                 19246.54
Name: 11, dtype: object

⚠️ Worst Performing Cohort:
CohortMonth               01-11-2011
Total_Customers                  263
Avg_AOV                       371.15
Avg_Purchase_Frequency          2.48
Avg_CLTV                      917.03
Avg_12Month_CLTV            11004.34
Avg_Max_CAC                   3301.3
Name: 10, dtype: object

📊 CLTV Gap between Best and Worst Cohort:
£53,150.79
